<a href="https://colab.research.google.com/github/Prasanna-Mahajan-2006/Deepfake-Detector/blob/main/MobileNet_%2B_FFT_(Notebook_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub

In [ ]:
import os
import cv2
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# 1. SETUP & DATASET LINKING
# ==========================================
print("Linking Dataset...")
# Ensure kagglehub has downloaded the dataset (it uses cache if already downloaded)
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")
DATASET_PATH = f"{path}/Dataset"
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 5

# ==========================================
# 2. DUAL-STREAM DATA GENERATOR
# ==========================================
class DualStreamGenerator(Sequence):
    def __init__(self, directory, batch_size=32, img_size=(224, 224), shuffle=True):
        self.directory = directory
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.filepaths = []
        self.labels = []

        for label, class_name in enumerate(['Real', 'Fake']):
            class_dir = os.path.join(directory, class_name)
            if os.path.exists(class_dir):
                for fname in os.listdir(class_dir):
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.filepaths.append(os.path.join(class_dir, fname))
                        self.labels.append(label)

        self.filepaths = np.array(self.filepaths)
        self.labels = np.array(self.labels)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.filepaths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_paths = self.filepaths[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_labels = self.labels[idx * self.batch_size:(idx + 1) * self.batch_size]

        X_spatial, X_freq, Y_valid = [], [], []

        for i, file_path in enumerate(batch_paths):
            img = cv2.imread(file_path)
            if img is None: continue

            # Spatial RGB
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_rgb = cv2.resize(img_rgb, self.img_size)
            img_rgb = img_rgb / 255.0

            # Frequency FFT
            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img_gray = cv2.resize(img_gray, self.img_size)
            f_shift = np.fft.fftshift(np.fft.fft2(img_gray))

            magnitude = 20 * np.log(np.abs(f_shift) + 1e-8)
            magnitude = (magnitude - np.min(magnitude)) / (np.max(magnitude) - np.min(magnitude) + 1e-8)
            magnitude = np.expand_dims(magnitude, axis=-1)

            X_spatial.append(img_rgb)
            X_freq.append(magnitude)
            Y_valid.append(batch_labels[i])

        return {
            "spatial_rgb_input": np.array(X_spatial),
            "frequency_fft_input": np.array(X_freq)
        }, np.array(Y_valid)

    def on_epoch_end(self):
        if self.shuffle:
            indices = np.arange(len(self.filepaths))
            np.random.shuffle(indices)
            self.filepaths = self.filepaths[indices]
            self.labels = self.labels[indices]

# ==========================================
# 3. HYBRID ARCHITECTURE (FINE-TUNING + FFT)
# ==========================================
def build_optimized_hybrid_model(img_size=(224, 224)):
    # --- STREAM 1: SPATIAL (Fine-Tuned MobileNet) ---
    spatial_input = layers.Input(shape=img_size + (3,), name="spatial_rgb_input")

    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=img_size + (3,), include_top=False, weights='imagenet'
    )

    # TRANSFER LEARNING: Unfreeze the top ~79 layers for active spatial learning
    base_model.trainable = True
    for layer in base_model.layers[:150]:
        layer.trainable = False

    x_spatial = base_model(spatial_input)
    x_spatial = layers.GlobalAveragePooling2D()(x_spatial)
    x_spatial = layers.BatchNormalization()(x_spatial)

    # --- STREAM 2: FREQUENCY (Deep FFT) ---
    freq_input = layers.Input(shape=img_size + (1,), name="frequency_fft_input")

    x_freq = layers.Conv2D(32, (3, 3), padding='same')(freq_input)
    x_freq = layers.BatchNormalization()(x_freq)
    x_freq = layers.Activation('relu')(x_freq)
    x_freq = layers.MaxPooling2D((2, 2))(x_freq)

    x_freq = layers.Conv2D(64, (3, 3), padding='same')(x_freq)
    x_freq = layers.BatchNormalization()(x_freq)
    x_freq = layers.Activation('relu')(x_freq)
    x_freq = layers.MaxPooling2D((2, 2))(x_freq)

    x_freq = layers.Conv2D(128, (3, 3), padding='same')(x_freq)
    x_freq = layers.BatchNormalization()(x_freq)
    x_freq = layers.Activation('relu')(x_freq)
    x_freq = layers.GlobalAveragePooling2D()(x_freq)

    # --- MERGE & REGULARIZE ---
    combined = layers.Concatenate()([x_spatial, x_freq])

    # REGULARIZATION: 50% Dropout and L2 penalty to prevent overfitting
    combined = layers.Dropout(0.5)(combined)
    outputs = layers.Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l2(0.01))(combined)

    model = models.Model(inputs=[spatial_input, freq_input], outputs=outputs, name="Optimized_Hybrid_Model")

    # Low learning rate (1e-4) is crucial because we are fine-tuning pre-trained weights
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
    )
    return model

# ==========================================
# 4. INITIALIZE & TRAIN
# ==========================================
print("Preparing Generators...")
train_gen = DualStreamGenerator(f"{DATASET_PATH}/Train", batch_size=BATCH_SIZE)
val_gen = DualStreamGenerator(f"{DATASET_PATH}/Validation", batch_size=BATCH_SIZE, shuffle=False)

model = build_optimized_hybrid_model()

callbacks = [
    keras.callbacks.ModelCheckpoint("best_final_model.keras", save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

print("\nStarting Training! (Watch for a much faster accuracy jump in Epoch 1 and 2)")
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

# ==========================================
# 5. FINAL EVALUATION & CONFUSION MATRIX
# ==========================================
print("\n--- Generating Final Evaluation Metrics ---")
print("Evaluating on Validation Data...")

# Get predictions from the unshuffled validation generator
y_pred_probs = model.predict(val_gen)
y_pred = (y_pred_probs > 0.5).astype("int32")
y_true = val_gen.labels

print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['Real (0)', 'Fake (1)']))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Final Hybrid Model Confusion Matrix')
plt.show()

Linking Dataset...
Using Colab cache for faster access to the 'deepfake-and-real-images' dataset.
Preparing Generators...

Starting Training! (Watch for a much faster accuracy jump in Epoch 1 and 2)
Epoch 1/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 2040s 463ms/step - accuracy: 0.7183 - loss: 0.5698 - precision: 0.7154 - recall: 0.7249 - val_accuracy: 0.5019 - val_loss: 4.6059 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 2/5
3466/4376 ━━━━━━━━━━━━━━━━━━━━ 2:47 184ms/step - accuracy: 0.7544 - loss: 0.5186 - precision: 0.7549 - recall: 0.7528

In [ ]:
# ==========================================
# 1. LOAD THE TRAINED MODEL
# ==========================================
MODEL_PATH = "best_final_model.keras" # Change this if you saved it under a different name
print(f"Loading trained model from {MODEL_PATH}...")
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Make sure the model file exists in your current directory.")

# ==========================================
# 2. IMAGE PREPROCESSING FUNCTION
# ==========================================
test_image_path = "/content/WhatsApp Image 2026-05-07 at 11.32.12.jpeg"
def preprocess_for_inference(test_image_path, img_size=(224, 224)):
    """
    Reads an image from disk and prepares the exact Dual-Stream
    input format (Spatial + FFT) expected by the hybrid model.
    """
    # 1. Read the image
    img = cv2.imread(test_image)
    if img is None:
        raise ValueError(f"Could not read image at path: {image_path}")

    # --- STREAM 1: SPATIAL DOMAIN (RGB) ---
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_rgb = cv2.resize(img_rgb, img_size)
    img_rgb = img_rgb / 255.0 # Normalize 0-1

    # Keras expects batches, so we add an extra dimension: (1, 224, 224, 3)
    spatial_input = np.expand_dims(img_rgb, axis=0)

    # --- STREAM 2: FREQUENCY DOMAIN (FFT) ---
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_gray = cv2.resize(img_gray, img_size)

    # Compute FFT and Shift
    f_shift = np.fft.fftshift(np.fft.fft2(img_gray))

    # Calculate Magnitude and Normalize
    magnitude = 20 * np.log(np.abs(f_shift) + 1e-8)
    magnitude = (magnitude - np.min(magnitude)) / (np.max(magnitude) - np.min(magnitude) + 1e-8)

    # Reshape for Keras: (1, 224, 224, 1)
    magnitude = np.expand_dims(magnitude, axis=-1)
    frequency_input = np.expand_dims(magnitude, axis=0)

    # Create the input dictionary exactly as the model expects
    input_dict = {
        "spatial_rgb_input": spatial_input,
        "frequency_fft_input": frequency_input
    }

    # Keep the original RGB array for displaying later
    return input_dict, img_rgb

# ==========================================
# 3. PREDICTION FUNCTION
# ==========================================
def analyze_image(image_path):
    """
    Runs an image through the model and prints the result with confidence levels.
    """
    print(f"\nAnalyzing Image: {image_path}")

    try:
        # Preprocess the image
        model_inputs, display_img = preprocess_for_inference(image_path)

        # Make the prediction
        # The model outputs a raw probability between 0.0 and 1.0
        prediction_prob = model.predict(model_inputs, verbose=0)[0][0]

        # Interpret the result (Assuming 0 = Real, 1 = Fake based on earlier logic)
        if prediction_prob > 0.5:
            classification = "FAKE"
            confidence = prediction_prob * 100
        else:
            classification = "REAL"
            # If the prob is 0.1, the model is 90% confident it's real
            confidence = (1.0 - prediction_prob) * 100

        # --- DISPLAY RESULTS ---
        print("-" * 30)
        print(f"Result: {classification}")
        print(f"Confidence: {confidence:.2f}%")
        print("-" * 30)

        # Show the image visually
        plt.figure(figsize=(4, 4))
        plt.imshow(display_img)
        plt.title(f"{classification} ({confidence:.1f}%)")
        plt.axis('off')
        plt.show()

    except Exception as e:
        print(f"An error occurred during analysis: {e}")

# ==========================================
# 4. RUN YOUR TEST!
# ==========================================
# Upload an image to your Colab environment and change this path
# Example: test_image_path = "/content/my_test_photo.jpg"

# For right now, let's just grab a random image from your existing validation set to test it!
test_image_path = f"{DATASET_PATH}/Validation/Fake/fake_100.jpg" # Update this to an actual file name in your folder

# Run the analysis
analyze_image(test_image_path)

In [ ]:
print("\n--- Generating Confusion Matrix & Evaluation Metrics ---")

# 1. Create an Unshuffled Generator
# CRUCIAL: shuffle=False ensures our predicted labels align perfectly with the true labels
print("Preparing unshuffled evaluation data...")
eval_gen = DualStreamGenerator(f"{DATASET_PATH}/Validation", batch_size=BATCH_SIZE, shuffle=False)

# 2. Generate Predictions
print("Running predictions on the validation set...")
y_pred_probs = regularized_model.predict(eval_gen)

# Convert probabilities to strict 0 (Real) or 1 (Fake) based on a 0.5 threshold
y_pred = (y_pred_probs > 0.5).astype("int32")

# 3. Extract True Labels directly from the generator
y_true = eval_gen.labels

# 4. Generate the Classification Report (Accuracy, Precision, Recall, F1-Score)
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['Real (0)', 'Fake (1)']))

# 5. Plot the Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])

plt.ylabel('Actual Label (Ground Truth)')
plt.xlabel('Predicted Label (Model Output)')
plt.title('Deepfake Detection Confusion Matrix (FFT Model)')
plt.show()


--- Generating Confusion Matrix & Evaluation Metrics ---
Preparing unshuffled evaluation data...


NameError: name 'DualStreamGenerator' is not defined